In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("/Users/macbookpro/Desktop/UAE_job_dashboard/data/raw/gsearch_jobs.csv")

In [3]:
# df.isnull().sum()
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
Unnamed: 0                 0
index                      0
title                      0
company_name               0
location                  13
via                        9
description                0
extensions                 0
job_id                     0
thumbnail               9985
posted_at                155
schedule_type            127
work_from_home         16401
salary                 25077
search_term                0
date_time                  0
search_location            0
commute_time           29621
salary_pay             25077
salary_rate            25077
salary_avg             25077
salary_min             25354
salary_max             25354
salary_hourly          27167
salary_yearly          27624
salary_standardized    25077
description_tokens         0
dtype: int64


In [4]:
# Drop columns we don't need
cols_to_drop = [
    'Unnamed: 0',    # just a row number
    'index',         # duplicate row number
    'thumbnail',     # image links, not useful
    'commute_time',  # 100% missing
    'salary',        # messy text, we have salary_standardized
    'salary_pay',    # duplicate
    'salary_rate',   # duplicate
    'salary_avg',    # duplicate
    'salary_min',    # too many missing
    'salary_max',    # too many missing
    'salary_hourly', # too many missing
    'salary_yearly', # too many missing
    'extensions',    # raw text not useful
    'job_id',        # not needed for analysis
]

df = df.drop(columns=cols_to_drop)

print("Remaining columns:", df.columns.tolist())
print("New shape:", df.shape)


Remaining columns: ['title', 'company_name', 'location', 'via', 'description', 'posted_at', 'schedule_type', 'work_from_home', 'search_term', 'date_time', 'search_location', 'salary_standardized', 'description_tokens']
New shape: (29621, 13)


In [5]:
# remove duplicates
# df = df.drop_duplicates()
print("After removing duplicates:", df.shape)

After removing duplicates: (29621, 13)


In [6]:
#fill missing values
df['work_from_home'] = df['work_from_home'].fillna(False)
df['location'] = df['location'].fillna('Unknown')
df['schedule_type'] = df['schedule_type'].fillna('Full-time')

print("Missing values left:")
print(df.isnull().sum())

Missing values left:
title                      0
company_name               0
location                   0
via                        9
description                0
posted_at                155
schedule_type              0
work_from_home             0
search_term                0
date_time                  0
search_location            0
salary_standardized    25077
description_tokens         0
dtype: int64


/var/folders/dg/n3npy9qd6m7fm_j210vlt_240000gn/T/ipykernel_1905/3060777702.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['work_from_home'] = df['work_from_home'].fillna(False)


In [7]:
#clean the skill column
import ast

def parse_skills(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

df['skills'] = df['description_tokens'].apply(parse_skills)

# Check it worked
print("Sample skills:")
print(df['skills'].head())

Sample skills:
0             [tableau, r, python, sql]
1                                    []
2                                 [sql]
3         [powerpoint, excel, power_bi]
4    [powerpoint, excel, outlook, word]
Name: skills, dtype: object


In [8]:
#save cleaned data
df.to_csv("../data/processed/jobs_cleaned.csv", index=False)
print("Saved to data/processed/!")
print("Clean data shape:", df.shape)
print("Step 2 complete!")

Saved to data/processed/!
Clean data shape: (29621, 14)
Step 2 complete!


In [9]:
# Load and check what is in jobs_cleaned.csv
df_clean = pd.read_csv("../data/processed/jobs_cleaned.csv")

print("Shape:", df_clean.shape)
print("\nColumns:", df_clean.columns.tolist())
print("\nFirst 3 rows:")
df_clean.head(3)

Shape: (29621, 14)

Columns: ['title', 'company_name', 'location', 'via', 'description', 'posted_at', 'schedule_type', 'work_from_home', 'search_term', 'date_time', 'search_location', 'salary_standardized', 'description_tokens', 'skills']

First 3 rows:


,title,company_name,location,via,description,posted_at,schedule_type,work_from_home,search_term,date_time,search_location,salary_standardized,description_tokens,skills
0,Data Analyst,Meta,Anywhere,via LinkedIn,In the intersection of compliance and analytic...,15 hours ago,Full-time,True,data analyst,2023-08-04 03:00:13.797776,United States,122000.0,"['tableau', 'r', 'python', 'sql']","['tableau', 'r', 'python', 'sql']"
1,Data Analyst,ATC,United States,via LinkedIn,Job Title: Entry Level Business Analyst / Prod...,12 hours ago,Full-time,False,data analyst,2023-08-04 03:00:13.797776,United States,NaN,[],[]
2,Aeronautical Data Analyst,"Garmin International, Inc.","Olathe, KS",via Indeed,Overview:\n\nWe are seeking a full-time...\nAe...,18 hours ago,Full-time,False,data analyst,2023-08-04 03:00:13.797776,United States,NaN,['sql'],['sql']
